# JSON 02 - A list of objects  (the shape you will actually meet)

Most real JSON is **an array of records**: an API returns a list of users, a
list of products, a list of tweets. In Python that is a `list` of `dict`.

Your data: **`../data/products.json`** - 8 products, each with
`id`, `name`, `category`, `price`, `stock`.

```json
[
  { "id": 1, "name": "Laptop", "category": "Computers", "price": 12000, "stock": 5 },
  ...
]
```

The top level is `[ ]`, so `json.load` gives you a **`list`**, not a dict. Which
means `data[0]` is the first product, and `for p in data:` walks all of them.

Everything below is normal Python on dicts and lists. There is no more "JSON"
after the `load` line - and that is the main thing to internalise.

## Exercise 0 - Load and look
Load `../data/products.json` into `products`.

- `type(products)` -> `<class 'list'>`
- `len(products)` -> **8**
- `products[0]["name"]` -> `Laptop`
- `type(products[0])` -> `<class 'dict'>`

In [19]:
import json
from dataclasses import dataclass, asdict
,
class Product:
    id: int
    name: str
    category: str
    price: int
    stock: int

class ProductDecoder:
    @staticmethod
    def decode(data: dict) -> Product:
        return Product(**data)

class ProductEncoder:
    @staticmethod
    def encode(product: Product) -> dict:
        return asdict(product)

class ProductRepository:
    def __init__(self, path: str):
        self.path = path
    def load(self) -> list[Product]:
        with open(self.path, "r") as file:
            data = json.load(file)
        return [
            ProductDecoder.decode(item)
            for item in data
        ]

    def save(self, products: list[Product]):
        data = [
            ProductEncoder.encode(product)
            for product in products
        ]
        with open(self.path, "w") as file:
            json.dump(
                data,
                file,
                indent=4
            )


## Exercise 1 - Total value of the stock

For each product the money sitting in the warehouse is `price * stock`.
Sum it for all products.

**Expected: 218700**

Do it twice:
- (a) with a normal `for` loop and an accumulator
- (b) with `sum(... for p in products)` on one line

Both are fine. Writing the loop first, then compressing it, is a good habit.

In [ ]:
# TODO

## Exercise 2 - The most expensive product

**Expected: `Laptop` at `12000`**

Do it twice:
- (a) loop keeping a `best` variable (like finding a max by hand)
- (b) `max(products, key=lambda p: p["price"])`

`key=` tells `max` *what number to compare*. It hands each dict to your little
function and compares what comes back. Same idea as `sorted(key=...)`.

In [ ]:
# TODO

## Exercise 3 - Filtering

Print the **names** of:

- all products in category `"Accessories"` -> `['Mouse', 'Keyboard', 'Charger', 'Headset']`
- all products with `stock > 20` -> `['Mouse', 'Keyboard', 'Charger']`

Try the list-comprehension form: `[p["name"] for p in products if ...]`

In [ ]:
# TODO

## Exercise 4 - Group by category  (this is the important one)

Build a dict `{ category: [names...] }`.

**Expected:**

```python
{
  'Computers':   ['Laptop', 'Monitor'],
  'Accessories': ['Mouse', 'Keyboard', 'Charger', 'Headset'],
  'Phones':      ['Phone', 'Tablet']
}
```

The problem: the first time you meet a category, the list does not exist yet, so
`groups[cat].append(...)` raises `KeyError`. Three ways to handle it - try at
least two:

1. `if cat not in groups: groups[cat] = []`
2. `groups.setdefault(cat, []).append(name)`
3. `from collections import defaultdict` ; `groups = defaultdict(list)`

Grouping records by a key is maybe the single most common thing you will ever do
with JSON data. Get comfortable here.

In [ ]:
# TODO

## Exercise 5 - Sorting

Print the names of the 3 most expensive products, most expensive first.

**Expected: `['Laptop', 'Phone', 'Tablet']`**

`sorted(products, key=lambda p: p["price"], reverse=True)` then slice `[:3]`.

Note `sorted` returns a **new list** and leaves `products` alone;
`products.sort()` changes it in place and returns `None`.

In [ ]:
# TODO

## Exercise 6 - Build an index  (Two Sum, again)

Right now, finding the product with `id == 5` means scanning the list: `O(n)`.
If you look things up many times, build a dict once:

`index = { p["id"]: p for p in products }`

Then `index[5]["name"]` is `O(1)` -> `Phone`.

Do it, and also build `by_name = { p["name"]: p }` so `by_name["Monitor"]["price"]`
gives `2200`.

This is exactly the lesson from Two Sum and from Exercise 4 of the Org Chart TP:
**when you search the same collection by the same key repeatedly, index it once.**

In [ ]:
# TODO

## Recap - 02

- Top-level `[` -> you get a `list`. Top-level `{` -> you get a `dict`. Check with `type()`.
- After loading, it is only dicts and lists. Loop, filter, `sum`, `max`, `sorted`.
- `key=lambda p: p["x"]` is how `max`/`sorted` compare records.
- `setdefault` / `defaultdict` for grouping.
- Index by key when you look up repeatedly.